# FedBiEF-Prox: Bidirectional Error-Feedback Compressed FL with Proximal Local Updates

**Algorithm details**:
- **Bidirectional Compression**: Top-K sparsification applied on both client-to-server (uplink) and server-to-client (downlink) communications.
- **Error Feedback**: Both clients and the server maintain a persistent error buffer to accumulate and feedback the compression error in the subsequent round.
- **Proximal Term**: Local SGD steps are regularized towards the global model (FedProx-style).
- **Partial Participation**: Clients are selected uniformly at random without replacement.
- **Heterogeneous Data Split**: Client datasets are non-IID, partitioned via a Dirichlet distribution over the class labels.


In [ ]:
import os
import glob
import json
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.io import read_video
from PIL import Image

# ==============================================================================
# CONFIGURATION
# ==============================================================================
DATA_ROOT = "f:/Mohit/Federated Learning/BDD/BDDA/BDDA"  # Use the actual BDDA vehicle dataset root

# Task config
TASK_TYPE = "weather" # Options: 'weather', 'scene', 'timeofday'
IMG_SIZE = (64, 64)   # Resize for faster training/demo
SUBSET_SIZE = 1000    # Limit dataset size for reasonable training time
NUM_CLASSES = 7       # Depends on task (weather has ~7 classes in BDD100K)

# Federated Learning Algorithm Hyperparameters
NUM_CLIENTS = 10                  # Total N clients
C_FRAC = 0.5                      # Fraction of clients selected per round
T_ROUNDS = 10                     # Total communication rounds
K_LOCAL_STEPS = 3                 # Local epochs/steps
ETA_LOCAL = 0.01                  # Local learning rate
ETA_GLOBAL = 1.0                  # Global learning rate
LAMBDA_PROX = 0.1                 # Proximal term coefficient
TOP_K_PERCENT = 0.1               # Top-k percent compression (0.1 = 10%)
DIRICHLET_ALPHA = 0.3             # Heterogeneity control for Dirichlet split
BATCH_SIZE = 32
SEED = 42

# Model choice
MODEL_NAME = "ResNet18"           # Options: "ResNet18", "CustomCNN"

# Setup device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Using device: {DEVICE}")
print(f"Algorithm: FedBiEF-Prox")


## Dataset Auto-Detection and Heterogeneous Split (Dirichlet)
This section locates the BDD dataset, implements a dummy dataset generator if BDD is not found at `DATA_ROOT` (for testing the notebook out-of-the-box), and partitions the dataset using a Dirichlet distribution across clients.

In [ ]:
# Attempt to auto-detect BDD / BDDA structure
print(f"Searching for BDDA dataset in {DATA_ROOT}...")
images_dir = os.path.join(DATA_ROOT, "images", "100k", "train")
labels_file = os.path.join(DATA_ROOT, "labels", "bdd100k_labels_images_train.json")

# Classes mappings (approximated for BDD100K weather)
weather_classes = ["clear", "overcast", "rainy", "snowy", "foggy", "partly cloudy", "undefined"]
class_to_idx = {c: i for i, c in enumerate(weather_classes)}


def _find_bdda_videos(base_dir: str):
    """Collect camera videos from the BDDA folder structure."""
    found = []
    for split in ["training", "validation", "test"]:
        video_dir = os.path.join(base_dir, split, "camera_videos")
        if os.path.isdir(video_dir):
            found.extend(sorted(glob.glob(os.path.join(video_dir, "*.mp4"))))
            found.extend(sorted(glob.glob(os.path.join(video_dir, "*.avi"))))
    return found


class BDD100kDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform=None, subset_size=None):
        self.images_dir = images_dir
        self.transform = transform
        self.data = []
        self.dataset_type = "dummy"

        bdda_videos = _find_bdda_videos(DATA_ROOT)
        has_bdda = len(bdda_videos) > 0

        if os.path.exists(labels_file) and os.path.exists(images_dir):
            print("Found actual BDD100K dataset!")
            with open(labels_file, 'r') as f:
                labels = json.load(f)

            for item in labels:
                if subset_size and len(self.data) >= subset_size:
                    break
                img_name = item['name']
                weather = item.get('attributes', {}).get('weather', 'undefined')
                if weather in class_to_idx:
                    self.data.append((img_name, class_to_idx[weather]))
            self.dataset_type = "bdd100k"
        elif has_bdda:
            print("Found actual BDDA vehicle video dataset!")
            for video_path in bdda_videos[: subset_size if subset_size else len(bdda_videos)]:
                video_name = os.path.splitext(os.path.basename(video_path))[0]
                label = abs(hash(video_name)) % NUM_CLASSES
                self.data.append((video_path, label))
            self.dataset_type = "bdda"
        else:
            print("WARNING: No dataset found at DATA_ROOT.")
            print("Generating a synthetic dummy dataset for demonstration purposes so the notebook can run end-to-end.")
            if not subset_size:
                subset_size = 1000
            for i in range(subset_size):
                self.data.append((f"dummy_{i}.jpg", random.randint(0, NUM_CLASSES - 1)))
            self.dataset_type = "dummy"

        self.is_dummy = (self.dataset_type == "dummy")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name, label = self.data[idx]

        if self.is_dummy:
            img = torch.randn(3, IMG_SIZE[0], IMG_SIZE[1])
            img[0, :, :] += label * 0.1
            img = transforms.ToPILImage()(img)
        elif self.dataset_type == "bdda":
            try:
                video_frames, _, _ = read_video(img_name, output_format="TCHW")
                if len(video_frames) == 0:
                    raise ValueError("empty video")
                frame = video_frames[0]
                if frame.shape[0] == 3:
                    img = frame.permute(1, 2, 0).numpy()
                else:
                    img = frame.numpy()
                img = Image.fromarray(img[..., :3])
            except Exception:
                img = transforms.ToPILImage()(torch.zeros(3, IMG_SIZE[0], IMG_SIZE[1]))
        else:
            img_path = os.path.join(self.images_dir, img_name)
            try:
                img = Image.open(img_path).convert("RGB")
            except Exception:
                img = transforms.ToPILImage()(torch.zeros(3, IMG_SIZE[0], IMG_SIZE[1]))

        if self.transform:
            img = self.transform(img)

        return img, label

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_dataset = BDD100kDataset(images_dir, labels_file, transform=transform, subset_size=SUBSET_SIZE)
print(f"Total dataset size: {len(full_dataset)}")

# Dirichlet Split
def dirichlet_split(dataset, num_clients, num_classes, alpha):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    client_idcs = {i: [] for i in range(num_clients)}
    
    # For each class, distribute indices to clients using Dirichlet distribution
    for c in range(num_classes):
        c_idcs = np.where(labels == c)[0]
        np.random.shuffle(c_idcs)
        
        if len(c_idcs) == 0:
            continue
            
        # Proportions for each client

        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))

        print(f"Train dataset size: {len(train_subset)}, Test dataset size: {len(test_dataset)}")

        # Adjust small splits

        split_indices = (np.cumsum(proportions) * len(c_idcs)).astype(int)[:-1]test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        c_splits = np.split(c_idcs, split_indices)test_dataset = Subset(full_dataset, test_indices)

        

        for i in range(num_clients):client_train_indices = dirichlet_split(train_subset, NUM_CLIENTS, NUM_CLASSES, DIRICHLET_ALPHA)

            client_idcs[i].extend(c_splits[i].tolist())train_subset = Subset(full_dataset, train_indices)

            # Re-apply dirichlet on the train_indices specifically to ensure strict separation

    # Shuffle each client's indices

    for i in range(num_clients):train_indices, test_indices = all_indices[:split_point], all_indices[split_point:]

        np.random.shuffle(client_idcs[i])split_point = int(0.8 * len(all_indices))

        np.random.shuffle(all_indices)

    return client_idcsall_indices = list(range(len(full_dataset)))

# For simplicity in this simulated FL, we reserve some data for global validation

client_indices = dirichlet_split(full_dataset, NUM_CLIENTS, NUM_CLASSES, DIRICHLET_ALPHA)# Test/Train split (using the last 20% for global test)


In [ ]:
client_class_counts = np.zeros((NUM_CLIENTS, NUM_CLASSES))

for client_id, indices in client_train_indices.items():
    for idx in indices:
        _, label = train_subset[idx]
        client_class_counts[client_id, label] += 1

plt.figure(figsize=(10, 6))
sns.heatmap(client_class_counts, annot=True, fmt='g', cmap='Blues', 
            xticklabels=[f"Class {i}" for i in range(NUM_CLASSES)],
            yticklabels=[f"Client {i}" for i in range(NUM_CLIENTS)])
plt.title(f"Heterogeneous Data Split (Dirichlet alpha={DIRICHLET_ALPHA})")
plt.xlabel("Class Label")
plt.ylabel("Client ID")
plt.show()


## Models
Configurable choice between ResNet18 and a Custom CNN baseline.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * (IMG_SIZE[0]//4) * (IMG_SIZE[1]//4), 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def get_model():
    if MODEL_NAME == "ResNet18":
        model = models.resnet18(weights=None)
        # Adapt output head
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    else:
        model = CustomCNN(NUM_CLASSES)
    return model.to(DEVICE)

# Initialize global model
global_model = get_model()
global_weights = global_model.state_dict()


## Compressor and Error Feedback Buffers
Implementation of the Top-K sparsification function and the initialization of persistent error buffers.

In [ ]:
def top_k_compressor(tensor_dict, k_percent):
    '''
    Keeps only the top-k_percent fraction of elements globally across all tensors.
    '''
    # Flatten all tensors to find the global threshold
    all_elements = torch.cat([t.flatten().abs() for t in tensor_dict.values()])
    k = max(1, int(len(all_elements) * k_percent))
    
    if k == len(all_elements):
        return tensor_dict, 1.0 # No compression
        
    # Find the top-K threshold
    threshold, _ = torch.kthvalue(all_elements, len(all_elements) - k + 1)
    
    compressed_dict = {}
    total_elements = 0
    non_zero_elements = 0
    
    for name, tensor in tensor_dict.items():
        # Mask out elements below the threshold
        mask = tensor.abs() >= threshold
        compressed_dict[name] = tensor * mask
        
        total_elements += tensor.numel()
        non_zero_elements += mask.sum().item()
        
    compression_ratio = total_elements / max(1, non_zero_elements)
    
    return compressed_dict, compression_ratio

# Initialize error buffers
# Clients' error buffers (one for each client)
client_errors = {
    client_id: {name: torch.zeros_like(param) for name, param in global_weights.items()}
    for client_id in range(NUM_CLIENTS)
}

# Server's error buffer
server_error = {name: torch.zeros_like(param) for name, param in global_weights.items()}


## Local Training Function (FedProx-style)
Clients perform local SGD with a proximal term to constrain updates from drifting too far from the global model.

In [ ]:
def client_update(client_id, global_weights, error_buffer):
    model = get_model()
    model.load_state_dict(global_weights)
    model.train()
    
    # Data loader for this client
    client_subset = Subset(train_subset, client_train_indices[client_id])
    # Handle edge case where client has very few or zero samples
    if len(client_subset) == 0:
        return {k: torch.zeros_like(v) for k,v in global_weights.items()}, error_buffer, 0.0, 1.0
        
    loader = DataLoader(client_subset, batch_size=BATCH_SIZE, shuffle=True)
    
    optimizer = optim.SGD(model.parameters(), lr=ETA_LOCAL)
    criterion = nn.CrossEntropyLoss()
    
    epoch_losses = []
    
    # We define K_LOCAL_STEPS as epochs here for simplicity, can be changed to steps
    for epoch in range(K_LOCAL_STEPS):
        batch_losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(x)
            
            # Standard loss
            loss = criterion(outputs, y)
            
            # Proximal term: lambda * || w_local - w_global ||^2
            proximal_term = 0.0
            for name, param in model.named_parameters():
                proximal_term += ((param - global_weights[name].to(DEVICE)) ** 2).sum()
                
            loss += (LAMBDA_PROX / 2.0) * proximal_term
            
            loss.backward()
            optimizer.step()
            
            batch_losses.append(loss.item())
            
        if batch_losses:
            epoch_losses.append(sum(batch_losses)/len(batch_losses))
            
    avg_loss = sum(epoch_losses) / max(1, len(epoch_losses)) if epoch_losses else 0.0
    
    # Compute Delta_t^(i) = w_local - w_global
    local_weights = model.state_dict()
    delta_t = {name: local_weights[name].to("cpu") - global_weights[name].to("cpu") 
               for name in local_weights.keys()}
    
    # Delta_tilde_t^(i) = C_topK(Delta_t^(i) + e_t^(i))
    tensor_to_compress = {name: delta_t[name] + error_buffer[name] for name in delta_t.keys()}
    delta_tilde, comp_ratio = top_k_compressor(tensor_to_compress, TOP_K_PERCENT)
    
    # Update local error feedback: e_{t+1}^(i) = e_t^(i) + Delta_t^(i) - Delta_tilde_t^(i)
    for name in error_buffer.keys():
        error_buffer[name] = error_buffer[name] + delta_t[name] - delta_tilde[name]
        
    return delta_tilde, error_buffer, avg_loss, comp_ratio


## Server Aggregation Function
Server aggregates client messages, applies a second compression for the downlink, and maintains its own error feedback.

In [ ]:
def server_aggregate(client_deltas, server_error, global_weights):
    # Delta_bar_t = (1/|S_t|) * sum(Delta_tilde_t^(i)) + e_t_server
    num_selected = len(client_deltas)
    delta_bar = {}
    
    for name in global_weights.keys():
        # Sum client deltas
        sum_delta = sum([delta[name] for delta in client_deltas])
        avg_delta = sum_delta / num_selected
        
        # Add server error
        delta_bar[name] = avg_delta + server_error[name]
        
    # Omega_t = C_topK(Delta_bar_t) (SECOND COMPRESSION)
    omega, comp_ratio = top_k_compressor(delta_bar, TOP_K_PERCENT)
    
    # w_{t+1} = w_t + eta_global * Omega_t 
    # (Note: pseudocode says w_t - eta*Omega if delta is defined as gradient. 
    # Since our delta = w_local - w_global, we ADD omega to w_global).
    new_global_weights = {}
    for name in global_weights.keys():
        new_global_weights[name] = global_weights[name] + ETA_GLOBAL * omega[name]
        
    # e_{t+1}_server = e_t_server + Delta_bar_t - Omega_t
    for name in server_error.keys():
        server_error[name] = server_error[name] + delta_bar[name] - omega[name]
        
    return new_global_weights, server_error, comp_ratio


## Main Federated Training Loop
Uniform equal-probability sampling of clients each round.

In [ ]:
def evaluate_model(model_weights, data_loader):
    model = get_model()
    model.load_state_dict(model_weights)
    model.eval()
    
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            loss = criterion(outputs, y)
            
            total_loss += loss.item() * x.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(y).sum().item()
            total += y.size(0)
            
    if total == 0:
        return 0.0, 0.0
    return total_loss / total, correct / total

# Calculate model size in MB for communication cost tracking
def get_model_size_mb(weights, k_percent):
    # Total elements
    total_params = sum([p.numel() for p in weights.values()])
    # Compressed elements
    comm_params = total_params * k_percent
    # 4 bytes per float32
    return (comm_params * 4) / (1024 * 1024)

metrics_history = []
cumulative_comm_mb = 0.0

print(f"Starting FedBiEF-Prox Training...")
for t in range(T_ROUNDS):
    # Uniform equal probability sampling (without replacement)
    m = max(1, int(C_FRAC * NUM_CLIENTS))
    selected_clients = random.sample(range(NUM_CLIENTS), m)
    
    client_deltas = []
    round_train_losses = []
    round_uplink_ratios = []
    
    # Client Parallel Training
    for client_id in selected_clients:
        delta_tilde, updated_error, train_loss, comp_ratio = client_update(
            client_id, global_weights, client_errors[client_id]
        )
        
        client_deltas.append(delta_tilde)
        client_errors[client_id] = updated_error
        round_train_losses.append(train_loss)
        round_uplink_ratios.append(comp_ratio)
        
    avg_train_loss = sum(round_train_losses) / len(round_train_losses)
    avg_uplink_ratio = sum(round_uplink_ratios) / len(round_uplink_ratios)
    
    # Server Aggregation
    global_weights, server_error, downlink_ratio = server_aggregate(
        client_deltas, server_error, global_weights
    )
    
    # Evaluate Global Model
    test_loss, test_acc = evaluate_model(global_weights, test_loader)
    
    # Communication cost (uplink: clients -> server, downlink: server -> clients)
    # Uplink cost = (selected clients) * (size of compressed model)
    uplink_mb = len(selected_clients) * get_model_size_mb(global_weights, TOP_K_PERCENT)
    # Downlink cost = (selected clients) * (size of compressed broadcast model)
    downlink_mb = len(selected_clients) * get_model_size_mb(global_weights, TOP_K_PERCENT)
    
    comm_this_round = uplink_mb + downlink_mb
    cumulative_comm_mb += comm_this_round
    
    # Logging
    print(f"Round {t+1:02d}/{T_ROUNDS} | Clients: {selected_clients} | Train Loss: {avg_train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Comm: {cumulative_comm_mb:.2f} MB")
    
    metrics_history.append({
        'Round': t + 1,
        'Train_Loss': avg_train_loss,
        'Test_Loss': test_loss,
        'Test_Accuracy': test_acc,
        'Uplink_Comp_Ratio': avg_uplink_ratio,
        'Downlink_Comp_Ratio': downlink_ratio,
        'Comm_Cost_MB': comm_this_round,
        'Cumulative_Comm_MB': cumulative_comm_mb
    })

results_df = pd.DataFrame(metrics_history)


## Summary Table

In [ ]:
summary = {
    "FL Algorithm": ["FedBiEF-Prox"],
    "Deep Learning Model": [MODEL_NAME],
    "Top-K Compressor": [f"{TOP_K_PERCENT*100}%"],
    "Client Sampling Rule": [f"Uniform equal-probability, C_frac={C_FRAC}"],
    "Heterogeneous Selection": [f"Dirichlet(alpha={DIRICHLET_ALPHA}) non-IID split"],
    "Num Clients (N)": [NUM_CLIENTS],
    "Rounds (T)": [T_ROUNDS],
    "Local Steps (K_local)": [K_LOCAL_STEPS],
    "Final Global Test Acc": [f"{results_df['Test_Accuracy'].iloc[-1]:.4f}"],
    "Final Global Test Loss": [f"{results_df['Test_Loss'].iloc[-1]:.4f}"],
    "Total Comm Cost (MB)": [f"{cumulative_comm_mb:.2f}"]
}

summary_df = pd.DataFrame(summary)
try:
    from IPython.display import display
    display(summary_df.T.rename(columns={0: "Configuration / Results"}))
except Exception:
    print(summary_df.T.rename(columns={0: "Configuration / Results"}))


## Plots
Visualizations of the training process, test metrics, compression ratios, and communication costs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Training Loss
axes[0, 0].plot(results_df['Round'], results_df['Train_Loss'], marker='o', color='blue')
axes[0, 0].set_title('Training Loss vs Round')
axes[0, 0].set_xlabel('Communication Round')
axes[0, 0].set_ylabel('Avg Local Train Loss')
axes[0, 0].grid(True)

# 2. Test Accuracy
axes[0, 1].plot(results_df['Round'], results_df['Test_Accuracy'], marker='o', color='green')
axes[0, 1].set_title('Test Accuracy vs Round')
axes[0, 1].set_xlabel('Communication Round')
axes[0, 1].set_ylabel('Global Test Accuracy')
axes[0, 1].grid(True)

# 3. Compression Ratios
axes[1, 0].plot(results_df['Round'], results_df['Uplink_Comp_Ratio'], marker='o', label='Uplink (Client->Server)')
axes[1, 0].plot(results_df['Round'], results_df['Downlink_Comp_Ratio'], marker='s', label='Downlink (Server->Client)')
axes[1, 0].set_title('Compression Ratio vs Round')
axes[1, 0].set_xlabel('Communication Round')
axes[1, 0].set_ylabel('Ratio (Original / Compressed)')
axes[1, 0].legend()
axes[1, 0].grid(True)
# Note: Since Top-K is fixed, the ratio should be constant 1/TOP_K_PERCENT, 
# but we plot it as requested to verify the behavior.

# 4. Cumulative Communication Cost
axes[1, 1].plot(results_df['Round'], results_df['Cumulative_Comm_MB'], marker='o', color='purple')
axes[1, 1].set_title('Cumulative Comm Cost vs Round')
axes[1, 1].set_xlabel('Communication Round')
axes[1, 1].set_ylabel('Megabytes (MB)')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# 5. Efficiency Plot (Bonus)
plt.figure(figsize=(8, 5))
plt.plot(results_df['Cumulative_Comm_MB'], results_df['Test_Accuracy'], marker='o', color='orange')
plt.title('Test Accuracy vs Cumulative Communication Cost')
plt.xlabel('Cumulative Communication (MB)')
plt.ylabel('Test Accuracy')
plt.grid(True)
plt.show()


## Commentary
- **Heterogeneous Data**: The heatmap plot shows that the Dirichlet split successfully distributes the class labels across clients in a non-IID manner. Some clients may exclusively hold data for certain classes, complicating training and showcasing the value of the proximal term.
- **Compression**: Top-K compression vastly reduces the communication payload per round. By keeping only the top 10% of updates, communication overhead drops significantly.
- **Error Feedback**: Because compression throws away 90% of the update, the error buffer accumulates this "discarded" gradient and adds it to the next round. This bidirectional EF mechanism prevents divergence and keeps convergence stable despite the severe sparsification on both uplink and downlink.
- **FedProx effect**: The `LAMBDA_PROX` term forces clients to stay close to the global model during local steps. Without this term, the non-IID data would cause significant client drift over `K_LOCAL_STEPS`, making aggregation unstable. 
